# LightGBM với 13 đặc trưng động và nhãn khởi phát v1

Upload `smoke_detection_iot_transient_v1.csv` thành Kaggle Dataset, chọn **Add Input** rồi **Run All**. Notebook tìm file trong `/kaggle/input` và hỗ trợ chạy cục bộ từ project.

Nhãn học là `Fire_Alarm_Transient`. Nhãn 1 biểu thị pha khởi phát PM đã chú giải. Nhãn 0 biểu thị ngoài cửa sổ khởi phát, bao gồm các giai đoạn sự kiện đang tiếp diễn. Nhãn hiện được suy ra từ cảm biến, chưa xác nhận bằng nhật ký cháy vật lý.

Notebook giữ đúng 13 đặc trưng. Chấm điểm bằng ROC AUC, Average Precision, log loss và Brier score. Điểm đầu ra mang nghĩa xác suất lớp khởi phát theo nhãn này. Tầng ngưỡng cảnh báo và biểu quyết được cấu hình ở hệ thống sử dụng model.

Validation luân phiên ba nhóm A/B/C, giữ các cặp phiên gần trùng cùng phía. Đây là đánh giá phát triển trên ba nhóm, chưa có tập test thực địa độc lập. Model cuối được fit lại trên toàn bộ các hàng đủ điều kiện sau khi hoàn thành validation.

In [1]:
from pathlib import Path
import hashlib
import json
import platform
import zipfile

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import (
    accuracy_score, average_precision_score, brier_score_loss, confusion_matrix,
    f1_score, log_loss, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import LeaveOneGroupOut

try:
    from IPython.display import FileLink, display
except ImportError:
    FileLink = None
    display = print

# Set a specific path here if the same filename occurs in several Kaggle Inputs.
DATASET_PATH = None
OUTPUT_DIR = Path('/kaggle/working/lightgbm13_transient_bundle') if Path('/kaggle/working').exists() else Path('lightgbm13_transient_bundle')
TARGET = 'Fire_Alarm_Transient'
CLASSIFICATION_THRESHOLD = 0.50
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 180)
FEATURES = [
    'd_temp_1', 'd_temp_5', 'd_hum_1', 'd_hum_5',
    'd_pm1_1', 'd_pm1_5', 'd_pm25_1', 'd_pm25_5',
    'temp_roll_5_std', 'hum_roll_5_std', 'pm1_roll_5_std',
    'pm25_roll_5_std', 'pm_ratio',
]
SENSORS = {
    'Temperature[C]': 'temp', 'Humidity[%]': 'hum',
    'PM1.0': 'pm1', 'PM2.5': 'pm25',
}
PARAMETERS = {
    'n_estimators': 150, 'max_depth': 6, 'num_leaves': 31,
    'learning_rate': 0.05, 'random_state': 42,
    'n_jobs': 2, 'verbosity': -1,
}
versions = {
    'python': platform.python_version(), 'lightgbm': lgb.__version__,
    'pandas': pd.__version__, 'numpy': np.__version__,
    'scikit_learn': sklearn.__version__, 'joblib': joblib.__version__,
}
print(json.dumps(versions, indent=2))


{
  "python": "3.12.16",
  "lightgbm": "4.6.0",
  "pandas": "2.2.3",
  "numpy": "2.2.6",
  "scikit_learn": "1.6.1",
  "joblib": "1.4.2"
}


## Nạp và kiểm tra dữ liệu

Các cột chú giải chỉ phục vụ chọn hàng, ghép kết quả và chia nhóm. Chỉ danh sách `FEATURES` được đưa vào LightGBM. Bộ dữ liệu v1 dự kiến có 62.630 hàng và 832 mẫu đích dương.

In [2]:
filename = 'smoke_detection_iot_transient_v1.csv'
if DATASET_PATH is None:
    candidates = []
    if Path('/kaggle/input').exists():
        candidates.extend(Path('/kaggle/input').rglob(filename))
    candidates.extend([
        Path.cwd() / filename,
        Path.cwd() / 'Model/smoke_transient_v1' / filename,
        Path.cwd().parent / 'Model/smoke_transient_v1' / filename,
    ])
    candidates = sorted({path.resolve() for path in candidates if path.is_file()})
    if len(candidates) != 1:
        raise ValueError(f'Expected one dataset. Set DATASET_PATH explicitly. Found {candidates}')
    dataset_path = candidates[0]
else:
    dataset_path = Path(DATASET_PATH)

dataset_sha256 = hashlib.sha256(dataset_path.read_bytes()).hexdigest()
df = pd.read_csv(dataset_path, float_precision='round_trip')
required = [*SENSORS, 'UTC', 'CNT', 'session_id', 'source_group', TARGET, 'training_eligible', 'label_version']
missing = sorted(set(required) - set(df.columns))
if missing:
    raise ValueError(f'Missing columns: {missing}')
if not df[TARGET].isin([0, 1]).all():
    raise ValueError('Target must contain binary 0/1 labels')
if not np.isfinite(df[[*SENSORS, 'UTC', 'CNT', 'session_id']].to_numpy(dtype=float)).all():
    raise ValueError('Invalid raw measurements or timestamps')

def read_boolean(series):
    if pd.api.types.is_bool_dtype(series):
        return series
    converted = series.astype(str).str.strip().str.lower().map({
        'true': True, 'false': False, '1': True, '0': False,
    })
    if converted.isna().any():
        raise ValueError(f'Invalid boolean values in {series.name}')
    return converted.astype(bool)

df['training_eligible'] = read_boolean(df['training_eligible'])
if 'needs_review' in df:
    df['needs_review'] = read_boolean(df['needs_review'])
    if (df['needs_review'] & df['training_eligible']).any():
        raise ValueError('Rows requiring annotation review were marked eligible')
expected_groups = {1: 'A', 5: 'A', 2: 'B', 3: 'B', 4: 'C'}
if not df.session_id.map(expected_groups).eq(df.source_group).all():
    raise ValueError('Source grouping differs from the reviewed five-session dataset')
if df.duplicated(['session_id', 'CNT']).any():
    raise ValueError('Duplicate sample keys')
if df.label_version.nunique() != 1:
    raise ValueError('Mixed annotation versions')

df = df.sort_values(['session_id', 'UTC', 'CNT']).reset_index(drop=True)
print('Dataset:', dataset_path)
print('SHA-256:', dataset_sha256)
print('Raw shape:', df.shape, '| target positives:', int(df[TARGET].sum()))


Dataset: /audit/Model/smoke_transient_v1/smoke_detection_iot_transient_v1.csv
SHA-256: 141056bbdd26ad78b0c4cec63ca665152d04a4f70fa95095d09d29c590a98159
Raw shape: (62630, 30) | target positives: 832


## Tính đúng 13 đặc trưng trước khi lọc hàng

Mỗi hàng dùng hiện tại và lịch sử. Chênh lệch 5 mẫu cần sáu mẫu liên tục. Cửa sổ độ lệch chuẩn dùng tối đa năm mẫu, `ddof=1`, cùng công thức backend. Reset lịch sử khi đổi phiên hoặc UTC/CNT không liên tục.

Sau khi tính trên toàn chuỗi, mới chọn `training_eligible=True` và đủ lịch sử. Các hàng bị loại khỏi loss vẫn giữ vai trò lịch sử cảm biến nếu chuỗi đo liên tục.

In [3]:
reset = (
    df.session_id.ne(df.session_id.shift())
    | df.UTC.diff().ne(1)
    | df.CNT.diff().ne(1)
)
block = reset.cumsum()
history_ready = df.groupby(block, sort=False).cumcount().ge(5)
X_full = pd.DataFrame(index=df.index)
for sensor, prefix in SENSORS.items():
    groups = df[sensor].groupby(block, sort=False)
    X_full[f'd_{prefix}_1'] = groups.diff(1).fillna(0.0)
    X_full[f'd_{prefix}_5'] = groups.diff(5).fillna(0.0)
    X_full[f'{prefix}_roll_5_std'] = groups.transform(
        lambda part: part.rolling(5, min_periods=1).std(ddof=1).fillna(0.0)
    )
X_full['pm_ratio'] = (df['PM2.5'] + 1e-3) / (df['PM1.0'] + 1e-3)
X_full = X_full.loc[:, FEATURES]
assert X_full.shape[1] == 13
if not np.isfinite(X_full.to_numpy()).all():
    raise ValueError('Nonfinite engineered features')

eligible = df.training_eligible & history_ready
X = X_full.loc[eligible].copy()
y = df.loc[eligible, TARGET].astype(int)
groups = df.loc[eligible, 'source_group']
group_table = df.loc[eligible].groupby('source_group')[TARGET].agg(rows='size', positive='sum')
group_table['negative'] = group_table['rows'] - group_table['positive']
if (group_table[['positive', 'negative']] == 0).any().any():
    raise ValueError('A validation group lacks one class')
print('Eligible features:', X.shape)
display(group_table)
display(X.head())


Eligible features: (62482, 13)
               rows  positive  negative
source_group                           
A             11478       658     10820
B             49896       108     49788
C              1108        66      1042
   d_temp_1  d_temp_5  d_hum_1  d_hum_5  d_pm1_1  d_pm1_5  d_pm25_1  d_pm25_5  temp_roll_5_std  hum_roll_5_std  pm1_roll_5_std  pm25_roll_5_std  pm_ratio
5     -0.05     -0.23     0.11     0.82     0.03     0.25      0.03      0.26         0.074364        0.211234        0.052631         0.055498  1.039284
6      0.00     -0.19     0.04     0.59     0.07     0.20      0.07      0.21         0.061806        0.158588        0.063008         0.064265  1.038119
7     -0.02     -0.16     0.07     0.44     0.05     0.22      0.05      0.22         0.048785        0.140961        0.071204         0.073824  1.037329
8     -0.04     -0.16     0.07     0.43     0.03     0.20      0.03      0.21         0.039749        0.110136        0.076616         0.076616  1.036870

## Validation theo nhóm và đánh giá điểm

Ba lượt giữ lần lượt nhóm A, B, C ngoài train. Mỗi hàng validation được chấm bởi model chưa học nhóm đó. Siêu tham số cố định trong cả ba lượt. Báo riêng điểm train/validation để thấy mức chênh lệch.

ROC AUC và Average Precision càng cao càng tốt. Log loss và Brier score càng thấp càng tốt. Average Precision cần đối chiếu với tỷ lệ dương của chính nhóm đó. OOF là các điểm ngoài fold từ ba model khác nhau; chỉ số gộp OOF được dùng bổ sung cho bảng từng nhóm.

In [4]:

def classification_metrics(truth, probability, threshold=CLASSIFICATION_THRESHOLD):
    predicted = (np.asarray(probability) >= threshold).astype(int)
    tn, fp, fn, tp = map(int, confusion_matrix(truth, predicted, labels=[0, 1]).ravel())
    return {
        'threshold': float(threshold), 'rows': int(len(truth)),
        'positive': int(np.sum(truth)),
        'accuracy': float(accuracy_score(truth, predicted)),
        'precision': float(precision_score(truth, predicted, zero_division=0)),
        'recall': float(recall_score(truth, predicted, zero_division=0)),
        'f1': float(f1_score(truth, predicted, zero_division=0)),
        'specificity': tn / (tn + fp) if tn + fp else None,
        'always_zero_accuracy': float(np.mean(np.asarray(truth) == 0)),
        'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp,
    }


def show_confusion_matrix(result, title):
    print(title)
    matrix = pd.DataFrame(
        [[result['TN'], result['FP']], [result['FN'], result['TP']]],
        index=['Thực tế 0', 'Thực tế 1'],
        columns=['Dự đoán 0', 'Dự đoán 1'],
    )
    display(matrix)


def score_metrics(truth, probability):
    return {
        'rows': int(len(truth)), 'positive': int(np.sum(truth)),
        'positive_rate': float(np.mean(truth)),
        'roc_auc': float(roc_auc_score(truth, probability)),
        'average_precision': float(average_precision_score(truth, probability)),
        'log_loss': float(log_loss(truth, probability, labels=[0, 1])),
        'brier_score': float(brier_score_loss(truth, probability)),
    }

splitter = LeaveOneGroupOut()
oof_probability = pd.Series(np.nan, index=X.index, dtype=float)
fold_assignment = pd.Series('', index=X.index, dtype=object)
metric_rows = []
classification_rows = []
for train_positions, val_positions in splitter.split(X, y, groups):
    train_index = X.index[train_positions]
    val_index = X.index[val_positions]
    held_group = str(groups.loc[val_index].iloc[0])
    assert set(groups.loc[train_index]).isdisjoint(set(groups.loc[val_index]))
    model = lgb.LGBMClassifier(**PARAMETERS)
    model.fit(X.loc[train_index], y.loc[train_index])
    positive_index = list(model.classes_).index(1)
    train_probability = model.predict_proba(X.loc[train_index])[:, positive_index]
    val_probability = model.predict_proba(X.loc[val_index])[:, positive_index]
    oof_probability.loc[val_index] = val_probability
    fold_assignment.loc[val_index] = held_group
    for subset, truth, probability in [
        ('train', y.loc[train_index], train_probability),
        ('validation', y.loc[val_index], val_probability),
    ]:
        metric_rows.append({'held_out_group': held_group, 'subset': subset,
                            **score_metrics(truth, probability)})
        classification_rows.append({'held_out_group': held_group, 'subset': subset,
                                    **classification_metrics(truth, probability)})
    print(f'Completed held-out group {held_group}')

assert oof_probability.notna().all()
fold_metrics = pd.DataFrame(metric_rows)
oof_metrics = score_metrics(y, oof_probability)
display(fold_metrics.round(6))
print('Supplementary pooled OOF metrics:')
display(pd.DataFrame([oof_metrics]).round(6))

validation_metrics = fold_metrics.loc[fold_metrics['subset'].eq('validation')]
print('Unweighted mean of three group validation results:')
display(validation_metrics[['roc_auc', 'average_precision', 'log_loss', 'brier_score']].mean().to_frame('mean'))


Completed held-out group A
Completed held-out group B
Completed held-out group C
  held_out_group      subset   rows  positive  positive_rate   roc_auc  average_precision  log_loss  brier_score
0              A       train  51004       174       0.003411  1.000000           1.000000  0.000545     0.000061
1              A  validation  11478       658       0.057327  0.780906           0.277139  0.411804     0.054853
2              B       train  12586       724       0.057524  0.999832           0.997430  0.018931     0.004282
3              B  validation  49896       108       0.002165  0.745794           0.182378  0.014845     0.001974
4              C       train  61374       766       0.012481  1.000000           0.999997  0.001423     0.000198
5              C  validation   1108        66       0.059567  0.803423           0.127943  0.888166     0.220614
Supplementary pooled OOF metrics:
    rows  positive  positive_rate   roc_auc  average_precision  log_loss  brier_score
0  62482

## Phân lớp tại ngưỡng 0,50 và confusion matrix

Cell này đổi điểm thành nhãn bằng `probability >= CLASSIFICATION_THRESHOLD` để tính accuracy, precision, recall và F1. Ngưỡng mặc định 0,50 phục vụ báo cáo phân lớp tức thời. Các chỉ số ROC AUC, Average Precision, log loss và Brier score phía trên vẫn được tính trực tiếp từ điểm model.

| | Dự đoán 0 | Dự đoán 1 |
|---|---|---|
| Thực tế 0 | TN — âm đúng | FP — dương giả |
| Thực tế 1 | FN — bỏ sót mẫu khởi phát | TP — dương đúng |

Accuracy = (TP + TN) / tổng mẫu. Precision = TP / (TP + FP). Recall = TP / (TP + FN). F1 kết hợp precision và recall.

Dữ liệu đủ điều kiện có khoảng 1,33% mẫu dương. Luôn dự đoán 0 đã đạt khoảng 98,67% accuracy, với recall bằng 0. Vì vậy, cần đọc accuracy cùng recall, precision, F1 và số FN/FP. `always_zero_accuracy` trong bảng cho biết mức đối chiếu này ở từng tập.


In [5]:
classification_table = pd.DataFrame(classification_rows)
classification_columns = [
    'held_out_group', 'subset', 'threshold', 'rows', 'positive',
    'accuracy', 'precision', 'recall', 'f1', 'specificity',
    'always_zero_accuracy', 'TN', 'FP', 'FN', 'TP',
]
print(f'Phân lớp tức thời với ngưỡng {CLASSIFICATION_THRESHOLD:.2f}')
display(classification_table[classification_columns].round(6))

for result in classification_rows:
    if result['subset'] == 'validation':
        show_confusion_matrix(result, f"Validation nhóm {result['held_out_group']}")

oof_classification = classification_metrics(y, oof_probability)
print('Phân lớp trên điểm OOF — mỗi hàng do model chưa học nhóm đó chấm:')
display(pd.DataFrame([oof_classification]).round(6))
show_confusion_matrix(oof_classification, 'Confusion matrix OOF gộp ba nhóm')


Phân lớp tức thời với ngưỡng 0.50
  held_out_group      subset  threshold   rows  positive  accuracy  precision    recall        f1  specificity  always_zero_accuracy     TN   FP   FN   TP
0              A       train        0.5  51004       174  1.000000   1.000000  1.000000  1.000000     1.000000              0.996589  50830    0    0  174
1              A  validation        0.5  11478       658  0.942412   0.465116  0.030395  0.057061     0.997874              0.942673  10797   23  638   20
2              B       train        0.5  12586       724  0.997457   0.987324  0.968232  0.977685     0.999241              0.942476  11853    9   23  701
3              B  validation        0.5  49896       108  0.997695   0.370370  0.092593  0.148148     0.999659              0.997835  49771   17   98   10
4              C       train        0.5  61374       766  0.999951   1.000000  0.996084  0.998038     1.000000              0.987519  60608    0    3  763
5              C  validation        

## Fit model cuối và xuất bundle chấm điểm

Model cuối sử dụng toàn bộ các hàng đủ điều kiện. Chỉ số train của model này được ghi riêng. Các điểm OOF vẫn đến từ ba model validation phía trên.

Bundle gồm model, metadata về mục tiêu và 13 đặc trưng, chỉ số từng fold và điểm OOF từng hàng. Metadata dành cho model khởi phát, cần được hệ thống suy luận đọc đúng ý nghĩa nhãn khi tích hợp.

In [6]:
final_model = lgb.LGBMClassifier(**PARAMETERS)
final_model.fit(X, y)
positive_index = list(final_model.classes_).index(1)
final_train_probability = final_model.predict_proba(X)[:, positive_index]
final_train_metrics = score_metrics(y, final_train_probability)
print('Final model training metrics:')
display(pd.DataFrame([final_train_metrics]).round(6))
final_train_classification = classification_metrics(y, final_train_probability)
print('Phân lớp trên dữ liệu đã dùng để fit model cuối:')
display(pd.DataFrame([final_train_classification]).round(6))
show_confusion_matrix(final_train_classification, 'Confusion matrix train model cuối')
comparison = pd.DataFrame([
    {'evaluation': 'OOF chưa học nhóm được chấm', **oof_classification},
    {'evaluation': 'Train model cuối', **final_train_classification},
])
print('Đối chiếu khả năng học train và chuyển sang nhóm ngoài train:')
display(comparison.round(6))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(final_model, OUTPUT_DIR / 'lightgbm13_transient_model.joblib')
metric_payload = {
    'folds': metric_rows, 'pooled_oof': oof_metrics,
    'final_model_train': final_train_metrics,
    'classification': {
        'threshold': CLASSIFICATION_THRESHOLD,
        'folds': classification_rows,
        'pooled_oof': oof_classification,
        'final_model_train': final_train_classification,
    },
}
metadata = {
    'model_name': 'LightGBM13_Transient_Onset_v1',
    'target': TARGET, 'positive_label': 1, 'classes': [int(v) for v in final_model.classes_],
    'features': FEATURES, 'n_features': len(FEATURES),
    'label_version': str(df.label_version.iloc[0]),
    'positive_meaning': 'Sensor-inferred PM onset window',
    'negative_meaning': 'Outside onset, including continuation and recovery',
    'physical_fire_ground_truth': 'unverified',
    'probability_calibration': 'not independently validated',
    'feature_extraction': {
        'history_samples': 6, 'rolling_window': 5, 'std_ddof': 1,
        'pm_ratio_epsilon': 1e-3, 'expected_sampling_seconds': 1,
        'reset_on_session_or_sample_gap': True,
    },
    'validation': {
        'method': 'LeaveOneGroupOut',
        'source_groups': {'A': [1, 5], 'B': [2, 3], 'C': [4]},
        'independent_field_test_available': False,
        'per_group_metrics_file': 'metrics.json',
        'classification_reporting_threshold': CLASSIFICATION_THRESHOLD,
    },
    'source': {'filename': dataset_path.name, 'sha256': dataset_sha256,
               'raw_rows': len(df), 'training_rows': len(X), 'training_positives': int(y.sum())},
    'parameters': PARAMETERS, 'versions': versions,
}
(OUTPUT_DIR / 'metadata.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2))
(OUTPUT_DIR / 'metrics.json').write_text(json.dumps(metric_payload, ensure_ascii=False, indent=2))

key_columns = [column for column in [
    'source_row_position', 'session_id', 'source_group', 'UTC', 'CNT',
    'transient_event_id', 'boundary_uncertain', TARGET,
] if column in df.columns]
oof_table = df.loc[X.index, key_columns].copy()
oof_table['held_out_group'] = fold_assignment
oof_table['oof_probability_onset'] = oof_probability
oof_table['oof_predicted_label'] = (oof_probability >= CLASSIFICATION_THRESHOLD).astype(int)
oof_table.to_csv(OUTPUT_DIR / 'oof_scores.csv', index=False)
fold_metrics.to_csv(OUTPUT_DIR / 'fold_metrics.csv', index=False)
classification_table.to_csv(OUTPUT_DIR / 'classification_fold_metrics.csv', index=False)
comparison.to_csv(OUTPUT_DIR / 'classification_summary.csv', index=False)

# Reload the exported estimator and check class/feature order and scoring parity.
restored = joblib.load(OUTPUT_DIR / 'lightgbm13_transient_model.joblib')
assert list(restored.feature_name_) == FEATURES
probe = X.iloc[:min(100, len(X))]
np.testing.assert_allclose(restored.predict_proba(probe), final_model.predict_proba(probe))
assert set(oof_table.held_out_group) == set(groups)

zip_path = OUTPUT_DIR.parent / 'lightgbm13_transient_bundle.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUTPUT_DIR.iterdir()):
        if path.is_file():
            archive.write(path, arcname=path.name)
print('Bundle:', zip_path)
if FileLink is not None:
    display(FileLink(str(zip_path)))


Final model training metrics:
    rows  positive  positive_rate   roc_auc  average_precision  log_loss  brier_score
0  62482       832       0.013316  0.999925           0.994866  0.005071     0.001156
Phân lớp trên dữ liệu đã dùng để fit model cuối:
   threshold   rows  positive  accuracy  precision    recall        f1  specificity  always_zero_accuracy     TN  FP  FN   TP
0        0.5  62482       832  0.999056   0.978934  0.949519  0.964002     0.999724              0.986684  61633  17  42  790
Confusion matrix train model cuối
           Dự đoán 0  Dự đoán 1
Thực tế 0      61633         17
Thực tế 1         42        790
Đối chiếu khả năng học train và chuyển sang nhóm ngoài train:
                    evaluation  threshold   rows  positive  accuracy  precision    recall        f1  specificity  always_zero_accuracy     TN   FP   FN   TP
0  OOF chưa học nhóm được chấm        0.5  62482       832  0.982891   0.201511  0.096154  0.130187     0.994858              0.986684  61333  317  

## Đọc kết quả

Bảng train của model cuối mô tả khả năng khớp dữ liệu đã học. Đánh giá khả năng chuyển sang nhóm mới cần xem các hàng validation A/B/C và kết quả OOF. AUC train gần 1 cùng recall hoặc F1 OOF thấp cho thấy mức chênh lớn giữa học train và tổng quát hóa.

Các bảng phân lớp dùng ngưỡng đã ghi rõ. Có thể thay `CLASSIFICATION_THRESHOLD` để khảo sát điểm vận hành rồi Run All. Việc chọn ngưỡng sau khi xem OOF biến OOF thành dữ liệu phát triển; cần dữ liệu giữ riêng mới để chấm cấu hình cuối. Hiệu năng ở đây được tính theo nhãn khởi phát cảm biến v1.
